In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip install torch  # torch is usually pre-installed
!pip install tqdm psutil wandb -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: leonn (leonn-texas-a-m-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import sqlite3
import math
import numpy as np
import torch
from torch.utils.data import Dataset


class MHeightDataset(Dataset):
    def __init__(self, n, k, m, db_path):
        self.n = n
        self.k = k
        self.m = m

        table_name = f"n-{n}_k-{k}_m-{m}"
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        rows = []
        try:
            cursor.execute(f'SELECT p_matrix, m_height FROM "{table_name}"')
            rows = cursor.fetchall()
        except sqlite3.OperationalError as e:
            print(f"Error reading from table {table_name}: {e}")
        finally:
            conn.close()

        N = len(rows)
        # Pre-allocate contiguous arrays — __getitem__ becomes a fast index op
        # instead of constructing a new tensor from a Python list each time.
        self.P_data = np.empty((N, k, n - k), dtype=np.float32)
        self.y_data = np.empty((N,),          dtype=np.float32)

        for i, (p_bytes, m_height) in enumerate(rows):
            self.P_data[i] = np.frombuffer(p_bytes, dtype=np.int8).reshape(k, n - k)
            self.y_data[i] = math.log2(max(1.0, float(m_height)))

    def __len__(self):
        return len(self.y_data)

    def __getitem__(self, idx):
        # torch.from_numpy on a pre-allocated array is zero-copy and much faster
        # than torch.tensor() on a Python list element.
        # Normalize to [-1, 1] — values are stored as integers in [-100, 100].
        P = torch.from_numpy(self.P_data[idx].copy()) / 100.0
        y = torch.tensor([self.y_data[idx]], dtype=torch.float32)
        return P, y


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.02):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = F.relu(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        return F.relu(out + x)


class DeepSetsNet(nn.Module):
    """
    Column-permutation-invariant network for estimating log2(m-height).

    Each column of P (a k-vector representing one parity position) is encoded
    independently by a shared MLP. The resulting embeddings are aggregated with
    sum + mean + max pooling, which is invariant to the order of the columns.
    Three scalar context values (n, k, m) are appended before the head.

    This architecture directly respects the mathematical symmetry that permuting
    the parity columns of G leaves the m-height of the code unchanged.
    """

    def __init__(self, k, n_minus_k, col_embed_dim=128, head_width=256, head_depth=4, dropout=0.02):
        super().__init__()

        # Shared encoder: maps each k-dimensional column → col_embed_dim embedding.
        # BatchNorm is applied over (batch * n_minus_k) samples, which is valid
        # because every column embedding is an independent forward pass through the same weights.
        self.col_encoder = nn.Sequential(
            nn.Linear(k, col_embed_dim),
            nn.BatchNorm1d(col_embed_dim),
            nn.ReLU(),
            nn.Linear(col_embed_dim, col_embed_dim),
            nn.BatchNorm1d(col_embed_dim),
            nn.ReLU(),
        )

        # sum + mean + max → 3 * col_embed_dim, then 3 context scalars (n, k, m)
        agg_dim = 3 * col_embed_dim + 3

        head_layers = [
            nn.Linear(agg_dim, head_width),
            nn.BatchNorm1d(head_width),
            nn.ReLU(),
        ]
        for _ in range(head_depth):
            head_layers.append(ResBlock(head_width, dropout))
        head_layers += [
            nn.Linear(head_width, head_width // 2),
            nn.ReLU(),
            nn.Linear(head_width // 2, 1),
        ]
        self.head = nn.Sequential(*head_layers)

    def forward(self, P, context):
        # P:       (batch, k, n_minus_k)
        # context: (batch, 3)  — [n, k, m] as floats
        batch, k, cols = P.shape

        # Flatten columns across the batch for the shared encoder
        P_cols = P.permute(0, 2, 1).reshape(batch * cols, k)  # (batch*cols, k)
        embeddings = self.col_encoder(P_cols)                  # (batch*cols, embed_dim)
        embeddings = embeddings.view(batch, cols, -1)          # (batch, cols, embed_dim)

        agg_sum  = embeddings.sum(dim=1)            # (batch, embed_dim)
        agg_mean = embeddings.mean(dim=1)           # (batch, embed_dim)
        agg_max  = embeddings.max(dim=1).values     # (batch, embed_dim)

        agg = torch.cat([agg_sum, agg_mean, agg_max, context], dim=1)
        return self.head(agg)


In [6]:
import os
import logging
import subprocess
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

try:
    import psutil
    PSUTIL = True
except ImportError:
    PSUTIL = False

try:
    import wandb
    WANDB = True
except ImportError:
    WANDB = False

DB_PATH     = "/content/drive/MyDrive/matrices.db"
SAVE_ROOT   = "/content/drive/MyDrive/ml_v3_models"
BATCH_SIZE  = 2048
NUM_WORKERS = min(8, os.cpu_count() or 4)
MODEL_TYPE  = "DeepSetsNet_v2"
logger      = logging.getLogger(__name__)


def augment_batch(P: torch.Tensor, n_minus_k: int) -> torch.Tensor:
    perm = torch.randperm(n_minus_k, device=P.device)
    P = P[:, :, perm]
    signs = torch.randint(0, 2, (1, 1, n_minus_k), device=P.device).float() * 2 - 1
    return P * signs


def gpu_stats() -> str:
    parts = []
    if torch.cuda.is_available():
        try:
            out = subprocess.check_output(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,utilization.memory",
                 "--format=csv,noheader,nounits"],
                timeout=2,
            ).decode().strip()
            gpu_util, mem_util = out.split(", ")
            parts.append(f"GPU {gpu_util}% | mem util {mem_util}%")
        except Exception:
            pass
        alloc = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        parts.append(f"VRAM {alloc:.1f}/{total:.1f} GB")
    if PSUTIL:
        parts.append(f"CPU {psutil.cpu_percent(interval=None):.0f}%")
        parts.append(f"RAM {psutil.virtual_memory().used / 1e9:.1f} GB")
    return "  ".join(parts) if parts else "no stats"


class EarlyStopping:
    def __init__(self, patience=15, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float("inf")
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss - val_loss > self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


def make_context(n, k, m, batch_size, device):
    ctx = torch.tensor([[float(n), float(k), float(m)]], dtype=torch.float32, device=device)
    return ctx.expand(batch_size, -1)


def evaluate(model, loader, criterion, n, k, m, device):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for P_batch, y_batch in loader:
            P_batch = P_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            ctx  = make_context(n, k, m, P_batch.size(0), device)
            pred = model(P_batch, ctx)
            total += criterion(pred, y_batch).item() * P_batch.size(0)
    return total / len(loader.dataset)


def train_member(seed, n, k, m, train_ds, val_ds, width, depth, dropout, device, save_dir, sync_time):
    torch.manual_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    model = DeepSetsNet(
        k=k, n_minus_k=n - k, col_embed_dim=128,
        head_width=width, head_depth=depth, dropout=dropout,
    ).to(device)

    criterion  = nn.MSELoss()
    optimizer  = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    early_stop = EarlyStopping(patience=15)

    best_val  = float("inf")
    save_path = os.path.join(save_dir, f"model_seed_{seed}.pth")

    run = None
    if WANDB:
        run = wandb.init(
            project="636-mheight",
            name=f"{MODEL_TYPE}_n{n}_k{k}_m{m}_seed{seed}_{sync_time}",
            config={
                "model": MODEL_TYPE,
                "n": n, "k": k, "m": m, "seed": seed,
                "width": width, "depth": depth, "dropout": dropout,
                "batch_size": BATCH_SIZE, "sync_time": sync_time,
            },
            reinit=True,
        )

    for epoch in range(250):
        model.train()
        train_loss = 0.0
        t0 = time.time()

        for P_batch, y_batch in train_loader:
            P_batch = P_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            P_batch = augment_batch(P_batch, n - k)
            ctx     = make_context(n, k, m, P_batch.size(0), device)
            optimizer.zero_grad()
            loss = criterion(model(P_batch, ctx), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * P_batch.size(0)

        epoch_secs = time.time() - t0
        avg_train  = train_loss / len(train_ds)
        avg_val    = evaluate(model, val_loader, criterion, n, k, m, device)
        scheduler.step(avg_val)

        if avg_val < best_val:
            best_val = avg_val
            torch.save({k_: v.cpu() for k_, v in model.state_dict().items()}, save_path)
            logger.info(f"Seed {seed} | Epoch {epoch+1:03d} | New best — saved ({best_val:.6f})")

        if WANDB and run is not None:
            run.log({
                "train_mse":  avg_train,
                "val_mse":    avg_val,
                "best_val":   best_val,
                "epoch_secs": epoch_secs,
                "lr":         optimizer.param_groups[0]["lr"],
            }, step=epoch)

        if (epoch + 1) % 10 == 0:
            logger.info(
                f"Seed {seed} | Epoch {epoch+1:03d} | "
                f"Train: {avg_train:.6f} | Val: {avg_val:.6f} | "
                f"Best: {best_val:.6f} | {epoch_secs:.1f}s | {gpu_stats()}"
            )

        early_stop(avg_val)
        if early_stop.early_stop:
            logger.info(f"Seed {seed} | Early stop at epoch {epoch + 1}")
            break

    logger.info(f"Seed {seed} done. Best val MSE: {best_val:.6f}")

    if WANDB and run is not None:
        run.summary["best_val_mse"] = best_val
        run.finish()

    model.load_state_dict(torch.load(save_path, map_location=device))
    return model, best_val


def train_combo(n, k, m, sync_time, num_seeds=3, save_root=SAVE_ROOT):
    is_hard = (m == n - k)
    width   = 512 if is_hard else 256
    depth   = 6   if is_hard else 4
    dropout = 0.05 if is_hard else 0.02

    full_ds = MHeightDataset(n, k, m, DB_PATH)
    if len(full_ds) == 0:
        logger.error(f"No data for n={n}, k={k}, m={m}")
        return

    train_size = int(0.8 * len(full_ds))
    val_size   = int(0.1 * len(full_ds))
    test_size  = len(full_ds) - train_size - val_size
    train_ds, val_ds, test_ds = random_split(
        full_ds, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    save_dir = os.path.join(save_root, MODEL_TYPE, f"n{n}_k{k}_m{m}_{sync_time}")
    os.makedirs(save_dir, exist_ok=True)

    log_path = os.path.join(save_dir, "train.log")
    fh = logging.FileHandler(log_path)
    fh.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(fh)

    batches_per_epoch = (train_size + BATCH_SIZE - 1) // BATCH_SIZE
    logger.info(
        f"n={n}, k={k}, m={m} | device={device} | batch={BATCH_SIZE} | workers={NUM_WORKERS} | "
        f"train={train_size:,} val={val_size:,} test={test_size:,} | "
        f"~{batches_per_epoch:,} batches/epoch | {gpu_stats()}"
    )

    models = []
    for seed in range(num_seeds):
        logger.info(f"--- Training seed {seed + 1}/{num_seeds} for n={n}, k={k}, m={m} ---")
        model, _ = train_member(
            seed, n, k, m, train_ds, val_ds, width, depth, dropout, device, save_dir, sync_time
        )
        models.append(model)

    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
    criterion = nn.MSELoss()
    total     = 0.0

    for model in models:
        model.eval()

    with torch.no_grad():
        for P_batch, y_batch in test_loader:
            P_batch = P_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            ctx   = make_context(n, k, m, P_batch.size(0), device)
            preds = torch.stack([m_model(P_batch, ctx) for m_model in models]).mean(dim=0)
            total += criterion(preds, y_batch).item() * P_batch.size(0)

    ensemble_mse = total / len(test_ds)
    logger.info(f"=== Ensemble test MSE for n={n}, k={k}, m={m}: {ensemble_mse:.6f} ===")

    logger.removeHandler(fh)
    fh.close()


def predict(n, k, m, P_matrix, models, device=None):
    """
    Run ensemble inference on a single P matrix and return the predicted m-height.

    P_matrix : numpy array of shape (k, n-k) with integer values in [-100, 100]
    models   : list of trained DeepSetsNet instances (one per ensemble seed)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Apply the same normalization used during training
    P = torch.tensor(P_matrix, dtype=torch.float32) / 100.0
    P = P.unsqueeze(0).to(device)                      # (1, k, n-k)
    ctx = make_context(n, k, m, 1, device)

    for model in models:
        model.eval()

    with torch.no_grad():
        log_pred = torch.stack([m(P, ctx) for m in models]).mean(dim=0).item()

    # Convert log2 prediction back to m-height and ensure the result is at least 1
    return max(1.0, 2 ** log_pred)


In [ ]:
import sys; sys.path.insert(0, 'ml_v3')
from datetime import datetime
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True  # overrides any previously configured handlers
)

sync_time = datetime.now().strftime("%Y%m%d_%H%M%S")
for n, k, m in [
    (9,4,2),
    (9,4,3),
    (9,4,4),
    (9,4,5),
    (9,5,2),
    (9,5,3),
    (9,5,4),
    (9,6,2),
    (9,6,3)
]:
    train_combo(n, k, m, sync_time, num_seeds=1)

2026-04-24 00:56:06,953 - INFO - n=9, k=4, m=2 | device=cuda | batch=2048 | workers=8 | train=6,490,923 val=811,365 test=811,366 | ~3,170 batches/epoch | GPU 24% | mem util 2%  VRAM 0.0/42.4 GB  CPU 66%  RAM 7.5 GB
2026-04-24 00:56:06,955 - INFO - --- Training seed 1/1 for n=9, k=4, m=2 ---


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


2026-04-24 00:57:23,457 - INFO - Seed 0 | Epoch 001 | New best — saved (0.096506)
2026-04-24 00:58:30,183 - INFO - Seed 0 | Epoch 002 | New best — saved (0.091088)
2026-04-24 00:59:36,936 - INFO - Seed 0 | Epoch 003 | New best — saved (0.087439)
2026-04-24 01:00:43,042 - INFO - Seed 0 | Epoch 004 | New best — saved (0.085184)
2026-04-24 01:01:50,152 - INFO - Seed 0 | Epoch 005 | New best — saved (0.081384)
2026-04-24 01:02:56,875 - INFO - Seed 0 | Epoch 006 | New best — saved (0.079899)
2026-04-24 01:05:10,596 - INFO - Seed 0 | Epoch 008 | New best — saved (0.077499)
2026-04-24 01:06:17,489 - INFO - Seed 0 | Epoch 009 | New best — saved (0.076437)
2026-04-24 01:07:22,599 - INFO - Seed 0 | Epoch 010 | New best — saved (0.075481)
2026-04-24 01:07:22,628 - INFO - Seed 0 | Epoch 010 | Train: 0.077221 | Val: 0.075481 | Best: 0.075481 | 59.6s | GPU 37% | mem util 2%  VRAM 0.0/42.4 GB  CPU 96%  RAM 8.2 GB
2026-04-24 01:09:37,263 - INFO - Seed 0 | Epoch 012 | New best — saved (0.075199)
2026-0

best_val,█▇▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch_secs,▅▅█▇▄▆▃▆▄▆▆▇▃▆▆▅▅▄▅▄▂▆▇▅▆▆▅▅▄▆▄▆▇▆▃▄▄▃▂▁
lr,████████████▄▄▄▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_mse,█▆▆▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mse,█▇▆▅▄▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val,0.06125
best_val_mse,0.06125
epoch_secs,58.29332
lr,0.0
train_mse,0.06018
val_mse,0.06125


2026-04-24 03:13:54,575 - INFO - === Ensemble test MSE for n=9, k=4, m=2: 0.061371 ===
2026-04-24 03:14:06,057 - INFO - n=9, k=4, m=3 | device=cuda | batch=2048 | workers=8 | train=2,068,948 val=258,618 test=258,619 | ~1,011 batches/epoch | GPU 25% | mem util 2%  VRAM 0.0/42.4 GB  CPU 93%  RAM 4.7 GB
2026-04-24 03:14:06,059 - INFO - --- Training seed 1/1 for n=9, k=4, m=3 ---


...

2026-04-24 04:13:02,833 - INFO - === Ensemble test MSE for n=9, k=4, m=4: 0.478752 ===
2026-04-24 04:13:07,278 - INFO - n=9, k=4, m=5 | device=cuda | batch=2048 | workers=8 | train=885,503 val=110,687 test=110,689 | ~433 batches/epoch | GPU 24% | mem util 2%  VRAM 0.0/42.4 GB  CPU 90%  RAM 7.4 GB
2026-04-24 04:13:07,280 - INFO - --- Training seed 1/1 for n=9, k=4, m=5 ---


2026-04-24 04:13:24,979 - INFO - Seed 0 | Epoch 001 | New best — saved (2.689470)
2026-04-24 04:13:34,967 - INFO - Seed 0 | Epoch 002 | New best — saved (2.667813)
2026-04-24 04:13:46,368 - INFO - Seed 0 | Epoch 003 | New best — saved (2.645125)
2026-04-24 04:13:57,158 - INFO - Seed 0 | Epoch 004 | New best — saved (2.621424)
2026-04-24 04:14:49,123 - INFO - Seed 0 | Epoch 009 | New best — saved (2.603551)
2026-04-24 04:15:00,061 - INFO - Seed 0 | Epoch 010 | New best — saved (2.592264)
2026-04-24 04:15:00,095 - INFO - Seed 0 | Epoch 010 | Train: 2.633420 | Val: 2.592264 | Best: 2.592264 | 9.8s | GPU 27% | mem util 1%  VRAM 0.1/42.4 GB  CPU 90%  RAM 7.4 GB
2026-04-24 04:15:10,462 - INFO - Seed 0 | Epoch 011 | New best — saved (2.590203)
2026-04-24 04:15:32,544 - INFO - Seed 0 | Epoch 013 | New best — saved (2.588212)
2026-04-24 04:15:42,806 - INFO - Seed 0 | Epoch 014 | New best — saved (2.584046)
2026-04-24 04:16:35,387 - INFO - Seed 0 | Epoch 019 | New best — saved (2.583651)
2026-04

best_val,█▆▆▆▆▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch_secs,▂█▅▂▅▂█▄▃▄▃▃▅▂▅▅▄▄▆▄▅▄▃▃▃▇▂▃▅▁▅▄▃▄▅▅▆▃▂▃
lr,██████████████████████████▄▄▄▄▃▃▃▃▂▂▂▁▁▁
train_mse,█▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mse,█▇▆▅▄▄▃▃▄▃▃▃▇▂▂▃▃▂▂▂▂▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂
best_val,2.53483
best_val_mse,2.53483
epoch_secs,9.21885
lr,3e-05
train_mse,2.48648
val_mse,2.55207


2026-04-24 04:27:50,514 - INFO - === Ensemble test MSE for n=9, k=4, m=5: 2.581656 ===
2026-04-24 04:28:25,792 - INFO - n=9, k=5, m=2 | device=cuda | batch=2048 | workers=8 | train=6,198,656 val=774,832 test=774,833 | ~3,027 batches/epoch | GPU 2% | mem util 0%  VRAM 0.0/42.4 GB  CPU 76%  RAM 6.4 GB
2026-04-24 04:28:25,794 - INFO - --- Training seed 1/1 for n=9, k=5, m=2 ---


2026-04-24 04:29:34,539 - INFO - Seed 0 | Epoch 001 | New best — saved (0.123885)
2026-04-24 04:30:36,911 - INFO - Seed 0 | Epoch 002 | New best — saved (0.105477)
2026-04-24 04:31:38,118 - INFO - Seed 0 | Epoch 003 | New best — saved (0.101532)
2026-04-24 04:32:39,297 - INFO - Seed 0 | Epoch 004 | New best — saved (0.098481)
2026-04-24 04:33:42,333 - INFO - Seed 0 | Epoch 005 | New best — saved (0.096720)
2026-04-24 04:34:44,495 - INFO - Seed 0 | Epoch 006 | New best — saved (0.093188)
2026-04-24 04:36:48,122 - INFO - Seed 0 | Epoch 008 | New best — saved (0.089135)
2026-04-24 04:37:50,985 - INFO - Seed 0 | Epoch 009 | New best — saved (0.087567)
2026-04-24 04:38:51,547 - INFO - Seed 0 | Epoch 010 | Train: 0.089779 | Val: 0.087853 | Best: 0.087567 | 55.0s | GPU 21% | mem util 0%  VRAM 0.0/42.4 GB  CPU 96%  RAM 6.1 GB
2026-04-24 04:39:54,857 - INFO - Seed 0 | Epoch 011 | New best — saved (0.087309)
2026-04-24 04:40:57,077 - INFO - Seed 0 | Epoch 012 | New best — saved (0.084612)
2026-0